# Demo: Flujo de Delegación del Orquestador Multi-Agente

Este notebook demuestra el flujo completo de delegación jerárquica:

```
Supervisor -> Investigador -> Supervisor -> Analista -> Supervisor -> FINISH
```

**Antes de correr esto**, asegurate de:
1. Tener las dependencias instaladas (`pip install -r requirements.txt`)
2. Tener `.env` configurado (por default usa Ollama, corriendo local)
3. Si usás Ollama, que esté corriendo y con el modelo descargado (`ollama pull llama3.2`)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_core.messages import HumanMessage
from graph import build_graph

grafo = build_graph()
print("Grafo compilado. Nodos:", list(grafo.get_graph().nodes.keys()))

## Diagrama del grafo (Mermaid)

Se genera directamente de la estructura del grafo compilado (no necesita ningún LLM).

In [ ]:
print(grafo.get_graph().draw_mermaid())

## Consulta que requiere ambos dominios de especialización

Esta consulta obliga al Supervisor a delegar primero al Investigador (recolectar comentarios)
y después al Analista (procesar sentimiento + estadísticas) antes de poder cerrar la tarea.

In [ ]:
consulta = (
    "Necesito saber qué opinan los usuarios sobre el checkout: "
    "buscá sus comentarios, analizá el sentimiento general y calculá "
    "el rating promedio."
)

resultado = grafo.invoke(
    {"messages": [HumanMessage(content=consulta)], "contribuciones": {}, "pasos": 0},
    config={"recursion_limit": 25},
)

print("Pasos del supervisor:", resultado["pasos"])
print("Contribuciones:", list(resultado["contribuciones"].keys()))

## Traza completa de la delegación

Cada mensaje muestra qué agente lo generó — así se puede auditar exactamente
quién aportó qué información en el camino hacia la respuesta final.

In [ ]:
for m in resultado["messages"]:
    agente = getattr(m, "name", None) or "usuario"
    print(f"[{agente}] {m.content}\n")

## Respuesta final sintetizada

El último mensaje es la síntesis del Supervisor, basada únicamente en las
contribuciones de ambos especialistas (no inventa datos nuevos).

In [ ]:
print(resultado["messages"][-1].content)